# Aluminosilicate Glass with SHIK Potential 

A LAMMPS script used to create a lithium aluminosilicate glass. The SHIK potential is used with a slight modification of using DSF model for Coulomb instead of wolf used in the original SHIK. This should not lead to significant changes in the properties and is used here to show an example. 

Scientific robustness of the results is to be discussed at a later stage. 

POTFILES has the potential table generated for using SHIK parameters.

In [ ]:
#!/usr/bin/env python3
"""
Python driver for LAMMPS input script: Glass formation simulation
Converted from Achraf Atila's LAMMPS script (01/04/2025).
The run times are reduced for testing purposes. In this script the box length is chosen automatically based on a given density, creates atoms of different types,
defines their properties (mass, charge), and sets up the potential interactions between them. The potential is taken as the buckingham potential + r^-24 part of the SHIK potential
and the Coulomb part is dealt with using DSF model. In principle it should be similar to the wolf method used in the original SHIK.

The script also includes commands for thermalization, equilibration, and data output. This includes equilibration runs at high temperature (4000K) and low temperature (300K), 
as well as cooling steps in between.
The script is designed to be run in a Python environment with the LAMMPS library installed.

It is important to ensure that the necessary potential files (e.g., table_O_O.tbl, table_Si_Si.tbl, etc.). For now they are hard coded in the script, and the potential files should be made separatly from this script.
However, it is straightforward to modify the script to create the potential files on the fly. It requires some additional coding and testing.


This script is intended for educational and research purposes and should be used with caution.
It is recommended to validate the results and ensure that the simulation setup is appropriate for the specific research question being addressed.
The script is provided "as is" without any warranties or guarantees of any kind.
Please refer to the LAMMPS documentation and the original author's notes for more information on the simulation setup and parameters used in this script.
Author: Achraf Atila
Date: 01/04/2025


COMMENTS: Many things can be changed in this script, and made automatic such as: the creation of potential table, etc.
to be discussed in the future.

This simulation below took around 44 min on my laptop. it may be shorter or longer  on other machines.
"""
from lammps import lammps

# Initialize LAMMPS, suppressing screen output
lmp = lammps(cmdargs=["-screen", "none"])

# Basic settings
lmp.command("units metal")
lmp.command("dimension 3")
lmp.command("boundary p p p")
lmp.command("atom_style charge")

# Variables
l = 100.0
density = 2.33 * 0.7  # g/cm3
N_O = 594
N_Si = 228
N_Al = 66
N_X = 78

# Create box and atoms
lmp.command(f"region box block 0.0 {l} 0.0 {l} 0.0 {l}")
lmp.command("create_box 4 box")

lmp.command(f"create_atoms 1 random {N_O} 802021 box overlap 2.5 maxtry 100000")
lmp.command(f"create_atoms 2 random {N_Si} 345879 box overlap 2.5 maxtry 100000")
lmp.command(f"create_atoms 3 random {N_Al} 598122 box overlap 2.5 maxtry 100000")
lmp.command(f"create_atoms 4 random {N_X} 584752 box overlap 2.5 maxtry 100000")

# Masses
lmp.command("mass 1 15.999400")
lmp.command("mass 2 28.085500")
lmp.command("mass 3 26.981539")
lmp.command("mass 4 6.9410000")

# Charges
lmp.command("set type 1 charge -0.9381969696969698")
lmp.command("set type 2 charge 1.7755")
lmp.command("set type 3 charge 1.6334")
lmp.command("set type 4 charge 0.5727")

# Groups
lmp.command("group O type 1")
lmp.command("group Si type 2")
lmp.command("group Al type 3")
lmp.command("group X type 4")

# Compute counts and masses, convert to SI inside Python
NA = 6.02214076e23
mass_O = (15.999400 / NA) * N_O
mass_Si = (28.085500 / NA) * N_Si
mass_Al = (26.981539 / NA) * N_Al
mass_X = (6.9410000 / NA) * N_X
mass_total = mass_O + mass_Si + mass_Al + mass_X
volume = mass_total / density  # cm3
L_new = (volume) ** (1 / 3) * 1e8  # Convert cm3 to Angstrom box length
print(f"Total mass (g): {mass_total}")
print(f"Volume (cm3): {volume}")
print(f"Rescaled box length (Å): {L_new}")

# Resize box
lmp.command(
    f"change_box all x final 0 {L_new} y final 0 {L_new} z final 0 {L_new} boundary p p p remap units box"
)

# Neighbor settings
lmp.command("neighbor 2.0 bin")
lmp.command("neigh_modify every 10 delay 10 check yes cluster yes page 100000 one 4000")

# Pair interactions
rvdw = 10.0
lmp.command("pair_style hybrid/overlay coul/dsf 0.2 10.0 table spline 50000")
pairs = [
    (1, 1, "table_O_O.tbl"),
    (1, 2, "table_O_Si.tbl"),
    (1, 3, "table_O_Al.tbl"),
    (1, 4, "table_O_Li.tbl"),
    (2, 2, "table_Si_Si.tbl"),
    (2, 4, "table_Si_Li.tbl"),
    (3, 3, "table_Al_Al.tbl"),
    (4, 4, "table_Li_Li.tbl"),
]
for i, j, fn in pairs:
    lmp.command(f'pair_coeff {i} {j} table "POTFILES/{fn}" SHIK_Buck_r24 {rvdw}')
lmp.command("pair_coeff * * coul/dsf")
lmp.command("pair_modify shift yes")

# Time integration & initial velocity
lmp.command("timestep 0.001")
lmp.command("velocity all create 4000.0 12547 rot yes mom yes dist gaussian")

# Short thermalization
lmp.command("thermo 10")
lmp.command(
    "thermo_style custom step atoms temp etotal vol enthalpy pe density press lx ly lz"
)
lmp.command("thermo_modify flush yes")

lmp.command("fix 4 all nve/limit 0.5")
lmp.command("fix 3 all langevin 4000.0 4000.0 0.01 549483")
lmp.command("run 2000")
lmp.command("unfix 4")
lmp.command("unfix 3")

lmp.command("reset_timestep 0")

# Compute per-atom properties
lmp.command("compute atomKen all ke/atom")
lmp.command("compute peatom all pe/atom")


# --------- saving data to be averaged over time for the analysis ---------
lmp.command("variable T equal temp")
lmp.command("variable PE equal pe")
lmp.command("variable KE equal ke")
lmp.command("variable TE equal etotal")
lmp.command("variable H equal enthalpy")
lmp.command("variable D equal density")
lmp.command("variable V equal vol")
lmp.command("variable P equal press")


# Output averages
lmp.command(
    "fix temp_out all ave/time 10 100 1000 v_T v_PE v_KE v_TE v_H v_D v_V v_P file data.out"
)

lmp.command("thermo 1000")
lmp.command(
    "thermo_style custom step atoms temp etotal vol enthalpy pe density press lx ly lz fnorm fmax"
)
lmp.command("thermo_modify flush yes")

# Dumps and runs for different ensembles
lmp.command(
    "dump simdump all custom 10000 XAS_equilibraion_HT.*.dump id type xu yu zu vx vy vz c_peatom c_atomKen"
)

# NVT at 4000K
lmp.command("fix 1 all nvt temp 4000.0 4000.0 0.1 tchain 5")
lmp.command("run 50000")  # reduced for speed and test puprose only
lmp.command("unfix 1")

# NPT equilibration
lmp.command(
    "fix 1 all npt temp 4000.0 4000.0 0.1 tchain 5 iso 1000.0 1000.0 1.0 pchain 5"
)
lmp.command("run 100000")
lmp.command("unfix 1")
lmp.command("undump simdump")

# NPT cooling steps
lmp.command(
    "dump simdump all custom 10000 XAS_Cooling.*.dump id type xu yu zu vx vy vz c_peatom c_atomKen"
)

lmp.command(
    "fix 1 all npt temp 4000.0 3500.0 0.1 tchain 5 iso 1000.0 1000.0 1.0 pchain 5"
)
lmp.command("run 50000")
lmp.command("unfix 1")

lmp.command("fix 1 all npt temp 3500.0 300.0 0.1 tchain 5 iso 1000.0 0.0 1.0 pchain 5")
lmp.command("run 320000")
lmp.command("unfix 1")
lmp.command("undump simdump")

# Low-temperature runs
lmp.command(
    "dump simdump all custom 10000 XAS_equilibration_LT.*.dump id type xu yu zu vx vy vz c_peatom c_atomKen"
)

lmp.command("fix 1 all npt temp 300.0 300.0 0.1 tchain 5 iso 0.0 0.0 1.0 pchain 5")
lmp.command("run 200000")
lmp.command("unfix 1")
lmp.command("undump simdump")

lmp.command(
    "dump simdump all custom 10000 XAS_stats_LT.*.dump id type xu yu zu vx vy vz c_peatom c_atomKen"
)

# Final NVT at ambient
lmp.command("fix 1 all nvt temp 300.0 300.0 0.1 tchain 5")
lmp.command("run 100000")
lmp.command("unfix 1")

# Print summary
lmp.command('print "Simulation complete"')

# Close LAMMPS
lmp.close()

LAMMPS (29 Aug 2024 - Update 1)
Created orthogonal box = (0 0 0) to (100 100 100)
  1 by 1 by 1 MPI processor grid
Created 594 atoms
  using lattice units in orthogonal box = (0 0 0) to (100 100 100)
  create_atoms CPU = 0.002 seconds
Created 228 atoms
  using lattice units in orthogonal box = (0 0 0) to (100 100 100)
  create_atoms CPU = 0.001 seconds
Created 66 atoms
  using lattice units in orthogonal box = (0 0 0) to (100 100 100)
  create_atoms CPU = 0.001 seconds
Created 78 atoms
  using lattice units in orthogonal box = (0 0 0) to (100 100 100)
  create_atoms CPU = 0.001 seconds
Setting atom values ...
  594 settings made for charge
Setting atom values ...
  228 settings made for charge
Setting atom values ...
  66 settings made for charge
Setting atom values ...
  78 settings made for charge
594 atoms in group O
Total mass (g): 3.027049333533014e-20
Volume (cm3): 1.855946862987746e-20
Rescaled box length (Å): 26.47617111762902
228 atoms in group Si
66 atoms in group Al
78 atoms